# Regularization Cheat Sheet (scikit-learn)

A quick-reference notebook — copy/paste snippets and consult the tables while working on any regularized linear model project. Not exercise-based; run cells to see the snippets work on a tiny synthetic example, then adapt them to your own data.

## 1. The core idea

```
Total Loss = Data Loss (e.g. MSE, log-loss)  +  α · Penalty(coefficients)
```

| Penalty | Formula | Name(s) | Effect |
|---|---|---|---|
| L1 | α·Σ\|bⱼ\| | Lasso | Shrinks *and* zeroes coefficients → feature selection |
| L2 | α·Σbⱼ² | Ridge | Shrinks coefficients toward zero, rarely exactly zero |
| Elastic Net | α₁·Σ\|bⱼ\| + α₂·Σbⱼ² | — | Blend, controlled by `l1_ratio` (1=L1, 0=L2) |

- The **intercept `b0` is never penalized** (translation invariance).
- **Always scale features first** (`StandardScaler`) — the penalty operates on raw coefficient magnitude, which is unit-dependent.
- Regularization happens **during** training (it's baked into the loss function), not as a preprocessing step.

## 2. alpha vs. C — don't mix these up

| | Linear regression (`Ridge`, `Lasso`) | Logistic regression (`LogisticRegression`) |
|---|---|---|
| Hyperparameter name | `alpha` | `C` |
| Relationship | `alpha` directly scales the penalty | `C = 1/alpha` |
| Bigger value means | **stronger** regularization | **weaker** regularization |
| Typical search range | Lasso: ~1e-4 to 10. Ridge: ~1e-2 to 1e4 (squared penalty needs bigger values) | ~1e-3 to 1e2 |
| Default | `alpha=1.0` | `C=1.0` (i.e. `alpha=1.0` too, but framed inversely) |

> `LogisticRegression()` is **regularized by default** (`penalty='l2'`, `C=1.0`) — unlike `LinearRegression()`, which has no penalty at all.

## 3. Imports you'll almost always need

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV,
    LogisticRegression, LogisticRegressionCV,
)
from sklearn.metrics import (
    mean_squared_error, f1_score, accuracy_score,
    precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
print("Imports OK")

## 4. Scaling + split boilerplate

In [ ]:
# X_raw: DataFrame of features, y: Series/array of target
def scale_and_split(X_raw, y, test_size=0.2, random_state=42, stratify=True):
    scaler = StandardScaler().fit(X_raw)
    X = scaler.transform(X_raw)
    strat = y if stratify else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=strat
    )
    return X_train, X_test, y_train, y_test, scaler

print("scale_and_split defined")

## 5. Regression: Ridge / Lasso / ElasticNet

In [ ]:
# Manual alpha
ridge = Ridge(alpha=1.0).fit  # ridge.fit(X_train, y_train)
lasso = Lasso(alpha=0.1).fit  # lasso.fit(X_train, y_train)
enet  = ElasticNet(alpha=0.1, l1_ratio=0.5).fit

# Built-in CV versions (search alpha automatically)
alphas = np.logspace(-4, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5)          # ridge_cv.fit(X, y); ridge_cv.alpha_
lasso_cv = LassoCV(alphas=alphas, cv=5)          # lasso_cv.fit(X, y); lasso_cv.alpha_
enet_cv  = ElasticNetCV(alphas=alphas, l1_ratio=[.1,.3,.5,.7,.9], cv=5)

print("Reference only — call .fit(X_train, y_train) on the object you need")

## 6. Classification: LogisticRegression variants

In [ ]:
# No regularization
clf_none = LogisticRegression(penalty=None, max_iter=5000)

# L2 (default) — any solver works
clf_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=5000)

# L1 — requires liblinear or saga
clf_l1 = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=5000)

# Elastic Net — requires saga, needs l1_ratio
clf_enet = LogisticRegression(
    penalty='elasticnet', C=1.0, l1_ratio=0.5, solver='saga', max_iter=5000
)

# Built-in CV version — searches C (and l1_ratio for elasticnet) automatically
clf_l1_cv = LogisticRegressionCV(
    Cs=np.logspace(-2, 2, 100), cv=5, penalty='l1',
    solver='liblinear', scoring='f1', max_iter=5000
)
print("Reference only — call .fit(X_train, y_train)")

## 7. Solver compatibility table

| Solver | l2 | l1 | elasticnet | none | Notes |
|---|---|---|---|---|---|
| `lbfgs` (default) | ✅ | ❌ | ❌ | ✅ | Fast, good default for l2/none |
| `liblinear` | ✅ | ✅ | ❌ | ❌ | Good for small datasets, required for l1 unless using saga |
| `saga` | ✅ | ✅ | ✅ | ✅ | Only solver supporting elasticnet; scales to large data |
| `newton-cg`, `sag` | ✅ | ❌ | ❌ | ✅ | Rarely needed for typical labs |

## 8. GridSearchCV pattern (works for any of the above)

In [ ]:
def tune_C(estimator, X_train, y_train, c_range=(-4, 2), n=100, scoring='f1', cv=5):
    param_grid = {'C': np.logspace(*c_range, n)}
    gs = GridSearchCV(estimator, param_grid=param_grid, scoring=scoring, cv=cv)
    gs.fit(X_train, y_train)
    return gs

def tune_alpha(estimator, X_train, y_train, a_range=(-4, 4), n=100, scoring='neg_mean_squared_error', cv=5):
    param_grid = {'alpha': np.logspace(*a_range, n)}
    gs = GridSearchCV(estimator, param_grid=param_grid, scoring=scoring, cv=cv)
    gs.fit(X_train, y_train)
    return gs

print("tune_C / tune_alpha defined — remember: negate .best_score_ for neg_mean_squared_error scoring")

## 9. Reading GridSearchCV / *CV results

| Attribute | Meaning |
|---|---|
| `.best_params_` | dict of the winning hyperparameter(s) |
| `.best_score_` | mean CV score at the winning params (negate if `scoring='neg_*'`) |
| `.best_estimator_` | a fitted clone of the model at the best params |
| `.cv_results_` | full grid of per-fold results — great for plotting score vs. hyperparameter |
| `RidgeCV`/`LassoCV`/`LogisticRegressionCV` | use `.alpha_` / `.C_` instead of `.best_params_` |

## 10. Evaluation & plotting snippets

In [ ]:
def coef_plot(model, feature_names, title='Coefficients'):
    coef = pd.Series(model.coef_.ravel(), feature_names).sort_values()
    coef.plot(kind='bar', title=title)
    plt.axhline(0, color='k', linewidth=0.8)
    plt.tight_layout()
    plt.show()

def classification_report_row(name, model, X_test, y_test):
    pred = model.predict(X_test)
    row = {'Model': name, 'Accuracy': accuracy_score(y_test, pred),
           'Precision': precision_score(y_test, pred), 'Recall': recall_score(y_test, pred),
           'F1': f1_score(y_test, pred)}
    if hasattr(model, 'predict_proba'):
        row['ROC-AUC'] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    return row

print("coef_plot / classification_report_row defined")

## 11. Decision guide: which regularization to reach for?

| Situation | Recommendation |
|---|---|
| You want automatic feature selection / suspect many irrelevant features | **L1 (Lasso)** |
| You want to keep all features but reduce their influence, or features are all plausibly relevant | **L2 (Ridge)** |
| Features are highly correlated and L1's "arbitrary pick one" behavior worries you | **Elastic Net** |
| You just want a sane default with no tuning yet | scikit-learn's default `LogisticRegression()` (L2, `C=1.0`) or `Ridge(alpha=1.0)` |
| You need the model to be maximally interpretable / sparse for a report | **L1**, then inspect zeroed coefficients |
| Dataset is small and solver speed doesn't matter | Any solver works — prefer `liblinear` for L1 simplicity |
| Dataset is large / high-dimensional | `saga`, and consider `SGDClassifier` for scale |

## 12. Common pitfalls

- **Forgetting to scale** before regularizing → penalty unfairly punishes large-unit features.
- **Confusing `alpha` and `C` direction** — a classic source of "why did increasing my hyperparameter make things worse/better than expected" confusion.
- **Fitting the final CV model on the full dataset** (`X, y`) and then evaluating it on `X_test` — the test rows were seen during training, inflating scores. Fine for exploration, not for a final reported number.
- **Linear-spaced hyperparameter grids** instead of `np.logspace` — wastes most of the grid on the wrong order of magnitude.
- **Using `liblinear` for elasticnet** — it doesn't support it; use `saga`.
- **Interpreting L1's chosen features as causally important** — L1 picks *a* representative from correlated groups somewhat arbitrarily; it is not proof the zeroed features are irrelevant.